In [0]:
from pyspark.sql.types import * 
from pyspark.sql.functions import *


In [0]:
rides_schema = StructType([StructField('ride_id', StringType(), True), StructField('confirmation_number', StringType(), True), StructField('passenger_id', StringType(), True), StructField('driver_id', StringType(), True), StructField('vehicle_id', StringType(), True), StructField('pickup_location_id', StringType(), True), StructField('dropoff_location_id', StringType(), True), StructField('vehicle_type_id', LongType(), True), StructField('vehicle_make_id', LongType(), True), StructField('payment_method_id', LongType(), True), StructField('ride_status_id', LongType(), True), StructField('pickup_city_id', LongType(), True), StructField('dropoff_city_id', LongType(), True), StructField('cancellation_reason_id', LongType(), True), StructField('passenger_name', StringType(), True), StructField('passenger_email', StringType(), True), StructField('passenger_phone', StringType(), True), StructField('driver_name', StringType(), True), StructField('driver_rating', DoubleType(), True), StructField('driver_phone', StringType(), True), StructField('driver_license', StringType(), True), StructField('vehicle_model', StringType(), True), StructField('vehicle_color', StringType(), True), StructField('license_plate', StringType(), True), StructField('pickup_address', StringType(), True), StructField('pickup_latitude', DoubleType(), True), StructField('pickup_longitude', DoubleType(), True), StructField('dropoff_address', StringType(), True), StructField('dropoff_latitude', DoubleType(), True), StructField('dropoff_longitude', DoubleType(), True), StructField('distance_miles', DoubleType(), True), StructField('duration_minutes', LongType(), True), StructField('booking_timestamp', TimestampType(), True), StructField('pickup_timestamp', TimestampType(), True), StructField('dropoff_timestamp', TimestampType(), True), StructField('base_fare', DoubleType(), True), StructField('distance_fare', DoubleType(), True), StructField('time_fare', DoubleType(), True), StructField('surge_multiplier', DoubleType(), True), StructField('subtotal', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('total_fare', DoubleType(), True), StructField('rating', DoubleType(), True)])

In [0]:
df = spark.read.table("uber.bronze.rides_raw")
df_parsed = df.withColumn("parsed_rides",from_json(col("rides"),rides_schema))\
    .select("parsed_rides.*")
display(df_parsed)

ride_id,confirmation_number,passenger_id,driver_id,vehicle_id,pickup_location_id,dropoff_location_id,vehicle_type_id,vehicle_make_id,payment_method_id,ride_status_id,pickup_city_id,dropoff_city_id,cancellation_reason_id,passenger_name,passenger_email,passenger_phone,driver_name,driver_rating,driver_phone,driver_license,vehicle_model,vehicle_color,license_plate,pickup_address,pickup_latitude,pickup_longitude,dropoff_address,dropoff_latitude,dropoff_longitude,distance_miles,duration_minutes,booking_timestamp,pickup_timestamp,dropoff_timestamp,base_fare,distance_fare,time_fare,surge_multiplier,subtotal,tip_amount,total_fare,rating


In [0]:
jinja_config = [

   

    {
        "table" : "uber.bronze.stg_rides",
        "select" : "uber.bronze.stg_rides.*",
        "where": ""
        
    },{
        "table" : "uber.bronze.map_vehicle_makes",
        "select" : "vehicle_make",
        "where": "",
        "on": "uber.bronze.stg_rides.vehicle_make_id = map_vehicle_makes.vehicle_make_id "
        
    },
    {
        "table" : "uber.bronze.map_vehicle_types",
        "select" : "vehicle_type,description,base_rate,per_mile,per_minute",
        "where": "",
        "on": "uber.bronze.stg_rides.vehicle_type_id = map_vehicle_types.vehicle_type_id "

    },
    {
        "table" : "uber.bronze.map_payment_methods",
        "select" : "payment_method,is_card,requires_auth",
        "where": "",
        "on": "uber.bronze.stg_rides.payment_method_id = map_payment_methods.payment_method_id "

    },
    {
        "table" : "uber.bronze.map_ride_statuses",
        "select" : "ride_status,is_completed",
        "where": "",
        "on": "uber.bronze.stg_rides.ride_status_id = map_ride_statuses.ride_status_id "

    },
    {
        "table" : "uber.bronze.map_cities",
        "select" : "city,state,region,updated_at",
        "where": "",
        "on": "uber.bronze.stg_rides.pickup_city_id = map_cities.city_id"

    },
    {
        "table" : "uber.bronze.map_cancellation_reasons",
        "select" : "cancellation_reason",
        "where": "",
        "on": "uber.bronze.stg_rides.cancellation_reason_id = map_cancellation_reasons.cancellation_reason_id"

    }
    
]

In [0]:
%sql
select fact.ride_id,fact.base_fare,loc.region
from uber.bronze.fact fact
left join uber.bronze.dim_location as loc
on fact.pickup_city_id = loc.pickup_city_id

ride_id,base_fare,region
0001b468-05ee-4359-a5c8-cdc3dd8b43cd,2.5,Northeast
002768fd-9e7e-45e7-af04-402b9cefa7ce,2.5,West
005dda50-48cf-481c-8017-e8110e1632ed,2.5,Midwest
00822655-e274-4432-b952-e4517c5b29dc,2.5,Northeast
009bf792-9656-43bd-a89a-cf2ff85eac79,2.5,West
0119c283-55dd-4d7a-8473-f666c50ef606,2.5,South
011fa6a5-4945-428d-937f-0ce7c212b304,2.5,Northeast
01303427-c5e1-41cf-8c7d-a8c589d3f3c4,2.5,Northeast
014ebf3a-5b03-432d-9028-1bcca332bd6b,2.5,Northeast
01ee0bc4-66d5-47af-9959-355a9fe2a871,2.5,Southwest


In [0]:

from jinja2 import Template


jinja_str = """

    SELECT
        {% for config in jinja_config %}
            {{config.select}}
                {% if not loop.last %}    
                    ,
                {% endif %}
        {% endfor %}


    FROM 
        {% for config in jinja_config %}
            {% if loop.first %}
                {{config.table}}
            
            {% else %}
                LEFT JOIN {{config.table}} ON {{config.on}}
            {% endif %}
        {% endfor %}

    
        {% for config in jinja_config %}
            {% if loop.first %}
                 {% if config.where != "" %}
                WHERE
            {% endif%}
                {% endif%}
            {{config.where}}    
                {% if not loop.last %}
                    {% if config.where != "" %}
                    AND
                {% endif %}
                    {% endif %}
        {% endfor %}

"""


template = Template(jinja_str)
rendered_template= template.render(jinja_config=jinja_config)
print(rendered_template)



    SELECT
        
            uber.bronze.stg_rides.*
                    
                    ,
                
        
            vehicle_make
                    
                    ,
                
        
            vehicle_type,description,base_rate,per_mile,per_minute
                    
                    ,
                
        
            payment_method,is_card,requires_auth
                    
                    ,
                
        
            ride_status,is_completed
                    
                    ,
                
        
            city,state,region,updated_at
                    
                    ,
                
        
            cancellation_reason
                
        


    FROM 
        
            
                uber.bronze.stg_rides
            
            
        
            
                LEFT JOIN uber.bronze.map_vehicle_makes ON uber.bronze.stg_rides.vehicle_make_id = map_vehicle_makes.vehicle_make_i